# YOLO-UDD v2.0 — Ablation Study (Kaggle, ready to run)

This notebook trains the four ablation configurations and reports their mAP, so you can
**prove how much each module (especially TAFM) contributes**.

| Run | PSEM | SDWH | TAFM | Purpose |
|-----|------|------|------|---------|
| `baseline`  | off | off | off | your own baseline |
| `psem`      | on  | off | off | PSEM gain |
| `psem_sdwh` | on  | on  | off | SDWH gain |
| `full`      | on  | on  | on  | **TAFM gain** |

### Before you run — 2 things
1. **Settings (right panel) → Accelerator → GPU** (T4 ×2 or P100).
2. **+ Add Input → Datasets** → add your TrashCan dataset, then set `DATA_DIR` in the
   cell below to its path (something like `/kaggle/input/<your-dataset>/trashcan`).

Then: **Run All**. The first full pass takes a while — start with the sanity cell.


## 1. Check the GPU

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU — switch it on in Settings, or runs will be very slow.')


## 2. Get the code (clone your GitHub repo)

In [ ]:
%cd /kaggle/working
![ -d YOLO-UDD-v2.0 ] && rm -rf YOLO-UDD-v2.0
!git clone https://github.com/kshitijkhede/YOLO-UDD-v2.0.git
%cd YOLO-UDD-v2.0
!ls


## 3. Install dependencies
PyTorch is already on Kaggle, so we only add the few extras. `albumentations` is pinned to
a version compatible with the dataset augmentations, and `numpy<2` avoids a common Kaggle
binary-mismatch error.

> If a cell later fails with a NumPy/OpenCV error, do **Run → Restart & clear cell outputs**,
> then run the notebook again (the pins take effect after a restart).

In [ ]:
!pip install -q 'numpy<2' 'albumentations==1.3.1' opencv-python-headless pyyaml tqdm tensorboard
print('deps installed')


## 4. Point to your dataset  ⟵ EDIT THIS
Set `DATA_DIR` to the folder that contains either
`instances_train_trashcan.json` / `instances_val_trashcan.json`, or an `annotations/`
folder with `train.json` / `val.json`, plus the images. Then verify.

In [ ]:
# EDIT this to your dataset path (see the 'Input' panel on the right for the exact name):
DATA_DIR = '/kaggle/input/trashcan/trashcan'
EPOCHS   = 50     # 50 for iteration; use 100 for final numbers
BATCH    = 16     # drop to 8 if you hit out-of-memory

import os
print('DATA_DIR exists:', os.path.isdir(DATA_DIR))
!python scripts/verify_dataset.py --dataset-dir "$DATA_DIR" || echo 'verify_dataset reported issues (check the path / layout above)'


## 5. Sanity check first (a few minutes)
Train the full model for just 3 epochs. You want to see the **loss go down** and a
**non-zero mAP** appear — that confirms the whole pipeline (data → loss → backprop →
metric) works before you spend hours on the real runs.

In [ ]:
!python scripts/train.py --config configs/train_config.yaml --data-dir "$DATA_DIR" \
    --ablation full --epochs 3 --batch-size {BATCH} --save-dir runs/sanity


## 6. Run the four ablations
Each run trains `EPOCHS` epochs and saves to `runs/train/<name>/checkpoints/best.pt`.
This is the long cell. If the session times out, re-run it — training auto-resumes from
the last checkpoint in each folder.

> Tip: to run just one, comment out the others in the list.

In [ ]:
ABLATIONS = ['baseline', 'psem', 'psem_sdwh', 'full']
for ab in ABLATIONS:
    print('\n' + '='*70 + f'\nTRAINING: {ab}\n' + '='*70)
    !python scripts/train.py --config configs/train_config.yaml --data-dir "$DATA_DIR" \
        --ablation {ab} --epochs {EPOCHS} --batch-size {BATCH}


## 7. Evaluate each run and build the results table
This is the table for your thesis/viva: it shows the mAP each configuration reached, so
the jump from `psem_sdwh` to `full` is TAFM's measured contribution.

In [ ]:
import json, os, pandas as pd
rows = []
for ab in ABLATIONS:
    ckpt = f'runs/train/{ab}/checkpoints/best.pt'
    if not os.path.exists(ckpt):
        print(f'skip {ab}: no checkpoint yet'); continue
    !python scripts/evaluate.py --weights {ckpt} --data-dir "$DATA_DIR" --save-dir runs/eval/{ab}
    res_file = f'runs/eval/{ab}/evaluation_results.json'
    if os.path.exists(res_file):
        r = json.load(open(res_file)); r['config'] = ab; rows.append(r)

if rows:
    df = pd.DataFrame(rows).set_index('config')
    cols = [c for c in ['precision','recall','mAP@50','mAP@50:95','mAP@75','FPS'] if c in df.columns]
    display(df[cols])
    df.to_csv('runs/ablation_results.csv')
    print('\nSaved runs/ablation_results.csv')
else:
    print('No results yet — run the training cell first.')


## 8. Save your outputs
Kaggle keeps everything under `/kaggle/working` as the notebook's **Output** after you
**Save Version** (Commit). Download `runs/ablation_results.csv` and the `best.pt` files
from the Output tab, and commit the results back to GitHub from your own machine.

**Interpreting the table:** a clear rise across baseline → psem → psem_sdwh → full is the
evidence that each module helps; the final step (→ full) is TAFM's contribution — the
headline result of the project.